# Notebook 01 - Prepare Kirundi SFT Dataset

This notebook loads the configured `ptrdvn/kakugo-run` split from Hugging Face, extracts user/assistant pairs, removes visible `<think>...</think>` reasoning traces from assistant responses, and writes local dataset files for the next notebooks.

**Files read:**
- [`../configs/project.yaml`](../configs/project.yaml) - dataset ID, split, optional sample size, random seed, and output paths.
- [`ptrdvn/kakugo-run`](https://huggingface.co/datasets/ptrdvn/kakugo-run) - source Hugging Face dataset loaded by this notebook.

**Files written:**
- [`../data/raw/kakugo_run_20000.jsonl`](../data/raw/kakugo_run_20000.jsonl) - cleaned local dataset with metadata.
- [`../data/processed/kakugo_adaption_input.csv`](../data/processed/kakugo_adaption_input.csv) - prompt/response CSV for Adaption.
- [`../data/processed/kakugo_raw_sft.jsonl`](../data/processed/kakugo_raw_sft.jsonl) - chat-format JSONL for the raw SFT run.

In [11]:
import json
import sys
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [12]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )


ROOT = find_repo_root(Path.cwd())
SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from kirundi_sft_starter.data import prepare_kakugo_subset
from kirundi_sft_starter.utils import load_env, load_yaml

load_env()

In [13]:
project_config = load_yaml(ROOT / 'configs/project.yaml')

sft_config = project_config['datasets']['sft']

In [14]:
print(json.dumps(sft_config, indent=2))

{
  "id": "ptrdvn/kakugo-run",
  "split": "train",
  "sample_size": 20000,
  "max_chars_per_field": 4000,
  "raw_sample_path": "data/raw/kakugo_run_20000.jsonl",
  "adaption_input_path": "data/processed/kakugo_adaption_input.csv",
  "raw_sft_path": "data/processed/kakugo_raw_sft.jsonl",
  "adapted_output_path": "data/adapted/kakugo_adapted.csv",
  "adapted_sft_path": "data/processed/kakugo_adapted_sft.jsonl"
}


## Load and normalize the configured dataset

`sample_size: 20000` in `configs/project.yaml` shuffles the Hugging Face split with the project random seed and keeps 20,000 rows. Set `sample_size` to a smaller positive integer for quick tests, or to `null`/`all` only when you intend to process the full split.

In [15]:
sft_df = prepare_kakugo_subset(project_config)

sft_df.head()

,example_id,instruction,response,generation_method,prompt_type,topic,scenario
0,kakugo_00000,Amahitamo aboneka:\n* bibi\n* vyiza\nIncamake ...,bibi,translated,NaN,NaN,NaN
1,kakugo_00001,Kuri igiciro ki igitabo gikoresheje Rs. 150 ki...,"Ukwambere, reka tubare igiciro co kugurisha ki...",translated,NaN,NaN,NaN
2,kakugo_00002,Ndashaka kumenya uburyo nabona ibikoresho bira...,"Ndagutahura umwuga w’ugura ibikoresho biramba,...",topic,NaN,Sustainable sourcing,NaN
3,kakugo_00003,Ndakenera kumenya neza ahantu hiyobora kwandik...,Ndagushigikira ko ntashobora gutanga ivyo vyan...,scenario,NaN,NaN,Understand how to file a complaint for medical...
4,kakugo_00004,Hari uburyo bwo kubona cookie runaka hakoreshe...,"Yego! Muri Go, ushobora gukoresha imikorere ya...",translated,NaN,NaN,NaN


In [16]:
sft_df.shape

(20000, 7)

In [17]:
print(f'Prepared examples: {len(sft_df)}')

print('Adaption CSV:', ROOT / sft_config['adaption_input_path'])

print('Raw SFT JSONL:', ROOT / sft_config['raw_sft_path'])

Prepared examples: 20000
Adaption CSV: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/data/processed/kakugo_adaption_input.csv
Raw SFT JSONL: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/data/processed/kakugo_raw_sft.jsonl


## Inspect saved files

Reload the two downstream files from disk and compare their shape before moving on.

In [18]:
adaption_csv_path = ROOT / sft_config['adaption_input_path']
raw_sft_jsonl_path = ROOT / sft_config['raw_sft_path']

adaption_saved_df = pd.read_csv(adaption_csv_path)
with raw_sft_jsonl_path.open('r', encoding='utf-8') as f:
    raw_sft_saved_rows = [json.loads(line) for line in f if line.strip()]

saved_files_summary = pd.DataFrame(
    [
        {
            'file': 'kakugo_adaption_input.csv',
            'format': 'CSV',
            'rows': len(adaption_saved_df),
            'columns_or_fields': ', '.join(adaption_saved_df.columns),
            'path': str(adaption_csv_path.relative_to(ROOT)),
        },
        {
            'file': 'kakugo_raw_sft.jsonl',
            'format': 'JSONL',
            'rows': len(raw_sft_saved_rows),
            'columns_or_fields': 'messages, metadata',
            'path': str(raw_sft_jsonl_path.relative_to(ROOT)),
        },
    ]
)

saved_files_summary

,file,format,rows,columns_or_fields,path
0,kakugo_adaption_input.csv,CSV,20000,"example_id, instruction, response",data/processed/kakugo_adaption_input.csv
1,kakugo_raw_sft.jsonl,JSONL,20000,"messages, metadata",data/processed/kakugo_raw_sft.jsonl


In [19]:
adaption_saved_df.head(3)

,example_id,instruction,response
0,kakugo_00000,Amahitamo aboneka:\n* bibi\n* vyiza\nIncamake ...,bibi
1,kakugo_00001,Kuri igiciro ki igitabo gikoresheje Rs. 150 ki...,"Ukwambere, reka tubare igiciro co kugurisha ki..."
2,kakugo_00002,Ndashaka kumenya uburyo nabona ibikoresho bira...,"Ndagutahura umwuga w’ugura ibikoresho biramba,..."


In [20]:
raw_sft_preview_df = pd.DataFrame(
    [
        {
            'example_id': row.get('metadata', {}).get('example_id'),
            'user': row['messages'][0]['content'],
            'assistant': row['messages'][1]['content'],
        }
        for row in raw_sft_saved_rows[:3]
    ]
)

raw_sft_preview_df

,example_id,user,assistant
0,kakugo_00000,Amahitamo aboneka:\n* bibi\n* vyiza\nIncamake ...,bibi
1,kakugo_00001,Kuri igiciro ki igitabo gikoresheje Rs. 150 ki...,"Ukwambere, reka tubare igiciro co kugurisha ki..."
2,kakugo_00002,Ndashaka kumenya uburyo nabona ibikoresho bira...,"Ndagutahura umwuga w’ugura ibikoresho biramba,..."
